# Tomato Leaf Disease — Inference / Test Notebook

## 1) Imports

In [1]:
import json
from dataclasses import dataclass, field
from typing import List

import numpy as np
import tensorflow as tf
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

print("TF version:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices('GPU'))


TF version: 2.20.0
GPU available: []


2026-09-16 20:04:41.505294: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


## 2) Config


In [2]:
IMG_SIZE = (224, 224)

MODEL_PATH = "/kaggle/input/models/ghadagsme/best-transfer-tomato-disease/keras/default/1/mobilenetv2_stage1_best.keras"
CLASS_INDICES_PATH = "/kaggle/input/datasets/ghadagsme/classindecies-jsonfile/class_indices.json"


## 3) Load class indices + model

In [3]:
with open(CLASS_INDICES_PATH, "r", encoding="utf-8") as f:
    CLASS_INDICES = {int(k): v for k, v in json.load(f).items()}

print(CLASS_INDICES)


{0: 'Tomato_Bacterial_spot', 1: 'Tomato_Leaf_Mold', 2: 'Tomato_Septoria_leaf_spot', 3: 'Tomato_Spider_mites_Two_spotted_spider_mite', 4: 'Tomato__Target_Spot', 5: 'Tomato__Tomato_YellowLeaf__Curl_Virus', 6: 'Tomato__Tomato_mosaic_virus', 7: 'Tomato_healthy'}


In [4]:
def load_trained_model(model_path: str = MODEL_PATH):
    return tf.keras.models.load_model(model_path)

model = load_trained_model(MODEL_PATH)
model.summary()


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_3 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 8)              │         1,032 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,752,986 (10.50 MB)

 Trainable params: 165,000 (644.53 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

 Optimizer params: 330,002 (1.26 MB)

## 4) Predict function

In [5]:
def predict_image(model, image_path: str):
    """
    Loads an image, preprocesses it exactly like training (resize + MobileNetV2
    preprocess_input), runs prediction, and returns (predicted_class, confidence).
    """
    img = tf.keras.utils.load_img(image_path, target_size=IMG_SIZE)
    img_array = tf.keras.utils.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0)
    img_array = preprocess_input(img_array)

    preds = model.predict(img_array, verbose=0)[0]
    predicted_index = int(np.argmax(preds))
    confidence = float(preds[predicted_index])
    predicted_class = CLASS_INDICES[predicted_index]

    return predicted_class, confidence


## 5) Treatment Recommendation Engine

In [6]:
@dataclass
class TreatmentInfo:
    disease_name: str
    severity: str
    urgency: str
    immediate_actions: List[str] = field(default_factory=list)
    organic_treatment: List[str] = field(default_factory=list)
    chemical_treatment: List[str] = field(default_factory=list)
    prevention: List[str] = field(default_factory=list)


TREATMENT_DB = {
    "Tomato_Bacterial_spot": TreatmentInfo(
        disease_name="Bacterial Spot",
        severity="medium",
        urgency="Spreads quickly in warm, humid conditions \u2014 act within 2 days",
        immediate_actions=[
            "Remove clearly infected leaves and destroy/dispose away from the field",
            "Reduce overhead watering; switch to drip irrigation if possible",
        ],
        organic_treatment=[
            "Apply copper-based bactericides on a regular schedule",
            "Improve airflow between plants to lower humidity",
        ],
        chemical_treatment=["Copper + mancozeb mix, following locally recommended dosage"],
        prevention=[
            "Crop rotation \u2014 avoid planting tomatoes in the same soil two years running",
            "Use certified disease-free seeds/seedlings",
        ],
    ),
    "Tomato_Leaf_Mold": TreatmentInfo(
        disease_name="Leaf Mold",
        severity="low",
        urgency="Relatively slow-spreading but weakens the crop \u2014 act within a week",
        immediate_actions=["Improve ventilation immediately", "Lower ambient humidity around the plant"],
        organic_treatment=["Sulfur-based fungicide spray"],
        chemical_treatment=["Chlorothalonil-based fungicide if needed"],
        prevention=["Good greenhouse ventilation", "Avoid overly dense planting"],
    ),
    "Tomato_Septoria_leaf_spot": TreatmentInfo(
        disease_name="Septoria Leaf Spot",
        severity="medium",
        urgency="Spreads from lower leaves upward \u2014 act within 3-4 days",
        immediate_actions=["Remove infected lower leaves immediately", "Clear away any infected plant debris"],
        organic_treatment=["Regular copper spray every 7-10 days during humid seasons"],
        chemical_treatment=["Mancozeb or chlorothalonil per local dosage guidance"],
        prevention=["At least 2-year crop rotation away from tomatoes", "Mulch to prevent soil-splash spread"],
    ),
    "Tomato_Spider_mites_Two_spotted_spider_mite": TreatmentInfo(
        disease_name="Two-Spotted Spider Mite",
        severity="medium",
        urgency="A pest, not a fungus/bacteria \u2014 spreads fast in hot, dry weather, act within 2-3 days",
        immediate_actions=["Check the undersides of leaves to confirm", "Spray plant with strong water jet to remove mites mechanically"],
        organic_treatment=["Neem oil or insecticidal soap", "Release natural predators (predatory mites) if feasible"],
        chemical_treatment=["Specialized miticides \u2014 regular insecticides usually don't work on mites"],
        prevention=["Maintain reasonable humidity", "Regularly inspect leaf undersides"],
    ),
    "Tomato__Target_Spot": TreatmentInfo(
        disease_name="Target Spot",
        severity="medium",
        urgency="Spreads in high humidity \u2014 act within 3-4 days",
        immediate_actions=["Remove visibly infected leaves/fruit", "Reduce how long leaves stay wet"],
        organic_treatment=["Regular copper spray"],
        chemical_treatment=["Fungicides with azoxystrobin or chlorothalonil"],
        prevention=["Good ventilation and adequate plant spacing", "Remove surrounding weeds"],
    ),
    "Tomato__Tomato_YellowLeaf__Curl_Virus": TreatmentInfo(
        disease_name="Tomato Yellow Leaf Curl Virus",
        severity="high",
        urgency="Viral, transmitted by whiteflies \u2014 no direct cure, focus on stopping spread immediately",
        immediate_actions=["Remove the entire infected plant, roots included, immediately", "Dispose away from the rest of the crop"],
        organic_treatment=["No organic cure for the virus itself \u2014 focus on controlling the whitefly vector"],
        chemical_treatment=["Insecticides targeting whiteflies to prevent spread to other plants"],
        prevention=["Insect netting on greenhouses", "Virus-resistant varieties next planting", "Proactive whitefly control"],
    ),
    "Tomato__Tomato_mosaic_virus": TreatmentInfo(
        disease_name="Tomato Mosaic Virus",
        severity="high",
        urgency="Viral and highly contact-contagious \u2014 act immediately to prevent spread",
        immediate_actions=[
            "Remove the infected plant immediately, roots included",
            "Sterilize any tools that touched it before using on other plants",
            "Wash hands thoroughly after handling the infected plant",
        ],
        organic_treatment=["No cure for the virus itself \u2014 focus solely on preventing spread"],
        chemical_treatment=["No chemical cures the virus \u2014 focus on isolation and sanitation"],
        prevention=["Certified virus-free seeds", "Sterilize tools between plants", "Avoid smoking near plants (virus can transfer from tobacco)"],
    ),
    "Tomato_healthy": TreatmentInfo(
        disease_name="Healthy",
        severity="low",
        urgency="No action required",
        immediate_actions=["Continue routine monitoring"],
        organic_treatment=[],
        chemical_treatment=[],
        prevention=["Maintain current watering and fertilizing practices"],
    ),
}


def get_recommendation(predicted_class: str, confidence: float, low_conf_threshold: float = 0.6,
                        medium_conf_threshold: float = 0.85) -> dict:
    if predicted_class not in TREATMENT_DB:
        return {
            "status": "unknown_class",
            "message": "Class not found in the knowledge base \u2014 check the class name sent by the model.",
        }

    info = TREATMENT_DB[predicted_class]

    if confidence < low_conf_threshold:
        return {
            "status": "low_confidence",
            "predicted_class": predicted_class,
            "confidence": confidence,
            "message": (
                f"The model isn't confident enough (below {int(low_conf_threshold * 100)}%). "
                "Try a clearer photo or consult an expert before taking action."
            ),
        }

    if confidence >= medium_conf_threshold:
        confidence_note = "The model is highly confident."
    else:
        confidence_note = "Moderate confidence \u2014 visual confirmation is recommended before chemical treatment."

    return {
        "status": "ok",
        "predicted_class": predicted_class,
        "confidence": confidence,
        "confidence_note": confidence_note,
        "severity": info.severity,
        "disease_name": info.disease_name,
        "urgency": info.urgency,
        "immediate_actions": info.immediate_actions,
        "organic_treatment": info.organic_treatment,
        "chemical_treatment": info.chemical_treatment,
        "prevention": info.prevention,
    }


## Upload widget


In [7]:
import ipywidgets as widgets
from IPython.display import display

uploader = widgets.FileUpload(accept='image/*', multiple=False)
display(uploader)


FileUpload(value=(), accept='image/*', description='Upload')

In [14]:
if uploader.value:
    if isinstance(uploader.value, tuple):
        file_info = uploader.value[0]
        name, content = file_info['name'], file_info['content']
    else:
        name = list(uploader.value.keys())[0]
        content = uploader.value[name]['content']

    image_path = f"/kaggle/working/{name}"
    with open(image_path, 'wb') as f:
        f.write(content)

    print("Saved to:", image_path)
else:
    print("\u0644\u0633\u0647 \u0645\u0627\u0631\u0641\u0639\u062a\u064a\u0634 \u0635\u0648\u0631\u0629")


Saved to: /kaggle/working/Screenshot 2026-09-16 211401.png


## 7) result


In [15]:
predicted_class, confidence = predict_image(model, image_path)
recommendation = get_recommendation(predicted_class, confidence)

print("Predicted class:", predicted_class)
print("Confidence:", round(confidence, 4))
print()
print(json.dumps(recommendation, ensure_ascii=False, indent=2))


Predicted class: Tomato_Bacterial_spot
Confidence: 0.9907

{
  "status": "ok",
  "predicted_class": "Tomato_Bacterial_spot",
  "confidence": 0.9906789660453796,
  "confidence_note": "The model is highly confident.",
  "severity": "medium",
  "disease_name": "Bacterial Spot",
  "urgency": "Spreads quickly in warm, humid conditions — act within 2 days",
  "immediate_actions": [
    "Remove clearly infected leaves and destroy/dispose away from the field",
    "Reduce overhead watering; switch to drip irrigation if possible"
  ],
  "organic_treatment": [
    "Apply copper-based bactericides on a regular schedule",
    "Improve airflow between plants to lower humidity"
  ],
  "chemical_treatment": [
    "Copper + mancozeb mix, following locally recommended dosage"
  ],
  "prevention": [
    "Crop rotation — avoid planting tomatoes in the same soil two years running",
    "Use certified disease-free seeds/seedlings"
  ]
}
